# ۱. شناخت داده‌های جدید لیگ قهرمانان اروپا

در این نوت‌بوک ساختار سه فایل، کیفیت داده و محدودیت‌های اولیه بررسی می‌شود. هیچ داده‌ای هنوز تغییر نمی‌کند.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "DataSet2").is_dir())
ANALYSIS_ROOT = ROOT / "UCL_Analysis2"
OUTPUT = ANALYSIS_ROOT / "Output"
sys.path.insert(0, str(ANALYSIS_ROOT / "src"))
pd.set_option("display.max_columns", 100)
plt.style.use("seaborn-v0_8-whitegrid")
print("Repository root:", ROOT)


## معرفی فایل‌ها

- `UCL_AllTime.csv`: آمار تجمعی ۳۵۴ تیم در کل تاریخ.
- `UEFA Champions League 2004-2021.csv`: مسابقات تاریخ‌دار با مرحله و گروه.
- `ucl.csv`: مسابقات فصل‌های ۲۰۱۰ تا ۲۰۲۱، بدون تاریخ و مرحله.

In [ ]:
files = sorted((ROOT / "DataSet2").glob("*.csv"))
rows = []
raw_frames = {}
for path in files:
    frame = pd.read_csv(path)
    raw_frames[path.name] = frame
    rows.append({
        "file": path.name,
        "rows": len(frame),
        "columns": len(frame.columns),
        "missing_cells": int(frame.isna().sum().sum()),
        "exact_duplicate_rows": int(frame.duplicated().sum()),
    })
pd.DataFrame(rows)


In [ ]:
for name, frame in raw_frames.items():
    print(f"\n{name}: {frame.shape}")
    display(frame.head(3))
    display(frame.isna().sum().rename("missing").to_frame().query("missing > 0"))


## کنترل سازگاری جدول All-Time

ستون `Points` منبع، امتیاز فوتبالی نیست. در تمام ردیف‌ها با تفاضل گل برابر است؛ پس در ادامه کنار گذاشته و `3×Wins + Draws` بازسازی می‌شود.

In [ ]:
all_raw = raw_frames["UCL_AllTime.csv"]
checks = pd.Series({
    "W + D + L equals Matches": int(((all_raw.Wins + all_raw.Draws + all_raw.Losses) == all_raw.Matches).sum()),
    "GF - GA equals Goal_Difference": int(((all_raw.Goals_scored - all_raw.Goals_conceded) == all_raw.Goal_Difference).sum()),
    "source Points equals Goal_Difference": int((all_raw.Points == all_raw.Goal_Difference).sum()),
}, name="rows_passing")
display(checks.to_frame())
display(all_raw.loc[(all_raw.Wins + all_raw.Draws + all_raw.Losses) != all_raw.Matches,
                    ["Team", "Matches", "Wins", "Draws", "Losses"]])


## کنترل امتیازهای غیرعددی

۲۱ مسابقه دارای annotation مربوط به وقت اضافه/پنالتی هستند. عدد اول به‌عنوان گل ثبت‌شده مسابقه استخراج می‌شود؛ نتیجه ضربات پنالتی در هدف `Home Win` وارد نمی‌شود.

In [ ]:
detailed_raw = raw_frames["UEFA Champions League 2004-2021.csv"]
score_mask = (~detailed_raw.homeScore.astype(str).str.fullmatch(r"\d+")) | (~detailed_raw.awayscore.astype(str).str.fullmatch(r"\d+"))
print("Annotated score rows:", int(score_mask.sum()))
display(detailed_raw.loc[score_mask, ["date", "homeTeam", "homeScore", "awayteam", "awayscore", "round"]].head(10))


## نتیجه کیفیت داده

`ucl.csv` دقیقاً دو بار تکرار شده است: ۲۹۲۲ ردیف خام ولی ۱۴۶۱ ردیف یکتا. این تکرار قبل از هر محاسبه حذف می‌شود. مقدار خالی `group` نیز خطا نیست؛ مرحله‌های حذفی طبیعتاً گروه ندارند.